In [ ]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))

In [ ]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
# from tdc.multi_pred import DTI

In [ ]:
## params
active_c = '#008bfb' # "#3B85C1"
silent_c = '#ff0051'

# dropbox/system
PATENTS_RAW           = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/5_Literature and Patents/0_Companies/Pharma_SmallMolecule_Patents_MASTER.xlsx'
DROPBOX_PROT          = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/12_Proteomics/5_Inventory/CDDVault/Proteomics/GiorgioTamo/'
DROPBOX_ML            = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/4_Data_Sciences/15_ML/'
DROPBOX_VALIDATION    = DROPBOX_ML+'Daniela/20260518_ML_selected_genes_DF.csv' # contains for 52 highlighted genes validation data on top 5 compounds
DOWNLOADS             = '/mnt/c/Users/gtamo/Downloads'
# data
## proteomics:
RAW_PROTEOMICS_PATH   = 'data/MS/20260424_Proteomics_Database_CSV_Export.csv' # df_raw
CLEAN_PROTEOMICS_PATH = 'data/MS/20260429_CDD_MS_SilentActive.csv' # silent vs active
PX_20260520_CDDVAULT  = DROPBOX_ML+'Daniela/20260519_FBXOscreening/20260520_Proteomics_CDDVault_Export.csv'
PX_20260520_DB        = DROPBOX_ML+'Daniela/20260519_FBXOscreening/20260520_Proteomics_Database_CSV_Export.csv'

## other
CHEMLIB_PATH          = 'data/chemical_libs/20260430_SERAC_lib.csv' # smiles + compound -> used to compute ML features
OT_ROOT               = 'data/external/opentarget'
PHARMA_PATENT_CSV     = 'data/patent/20260512_pharma_sm.csv' # pharma targets of interest
# output
OT_CACHE              = 'output/MS/opentargets_target_disease.parquet'
GENE_SAR_OUT          = 'output/MS/20260518_geneSAR_R2_full_genome.csv' # R2 per gene
MCS_CSV               = 'output/MS/20260505_target_final_mcs.csv' # MCS enrichment
ML_MODEL_OUTPUT       = 'output/ML/trained_models/20260513' # dump for trained ML models

# misc
CM2RM                 = ['SRB-0005653']
FEATURES_TYPE         = 'prevalence' # 'autoresearch' # 
PHARMA_R2_CUTOFF      = 0.08 # above this R2 we show pharma relevant targets
MUSIC_SET             = 'output/enumeration/20260429/sdf/20260429_Music_1K.sdf'


## 0. Imports

In [ ]:
%%time
## raw proteomics data
df_raw   = pd.read_csv(RAW_PROTEOMICS_PATH)
parts = df_raw['MoleculeBatchID'].str.split('-', n=2, expand=True)
df_raw['compound'] = parts[0] + '-' + parts[1]   # 'SRB-0000385'
df_raw['batch']    = parts[2]                    # '001'
## compounds ids and smiles
serac_df = (pd.read_csv(CHEMLIB_PATH)
            .drop(['CDD Number','Synonyms'],axis=1)
            .rename(columns={'Molecule Name':'compound','SMILES':'smiles','Projects':'project'})
            .drop('project',axis=1)
            .drop_duplicates())

## Get clean MS data to get plate Ids
MS = pd.read_csv(CLEAN_PROTEOMICS_PATH).drop(['CDD Number'],axis=1)
MS = MS[MS['MSData - Proteomics activities: Source']=='SERAC'] # 
MS['MSData - Proteomics activities: Date'] = pd.to_datetime(MS['MSData - Proteomics activities: Date'])
MS['MSData - Proteomics activities: MSPlate'].unique()

# remove plate 12 data (from Daniela)
# MS = MS[~MS['MSData - Proteomics activities: MSPlate'].isin(['Pw33','Plate12']) ]

# if a compound is tested multiple times, get only by latest date
MS = MS.sort_values('MSData - Proteomics activities: Date',ascending=False).reset_index()
MS = MS.groupby('Molecule Name').first().reset_index()

print('>', len(MS['Molecule Name'].unique()),'unique compounds')
print('> Ligase(s)',list(MS['MSData - Proteomics activities: Ligase'].unique()))
print('> Cellline',list(MS['MSData - Proteomics activities: Cell line'].unique()))
print('> dim:',MS.shape)
MS.head(2)


In [ ]:
%%time
## Clean, filter and format raw MS data
df_raw = df_raw[~df_raw['MSPlate'].isin(['Plate12','Plate15','Plate23']) ]
df_raw = df_raw[df_raw['MoleculeBatchID'].isin(MS['MSData - Proteomics activities: Molecule-Batch ID'])]
df_raw.head(2)